In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pickle

In [19]:
with open("training_data.txt", "r", encoding="utf-8") as f :
    text = f.read()

print('characters : ', len(text))
print()
print(text[:300])

characters :  1420167

BOOK I.
The quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mother Thetis to ask Jove to help the Trojans—Scene between Jove and Juno on Olympus.

Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans. Many a brave s


In [13]:
with open("bpe_tokenizer.pkl", "rb") as f:
    tokenizer_data = pickle.load(f)

merges = tokenizer_data["merges"]
bpe_stoi = tokenizer_data["bpe_stoi"]
bpe_itos = tokenizer_data["bpe_itos"]
bpe_vocab = tokenizer_data["bpe_vocab"]
bpe_vocab_size = tokenizer_data["bpe_vocab_size"]

print("Number of merges:", len(merges))
print("Vocabulary size:", bpe_vocab_size)

Number of merges: 1000
Vocabulary size: 1080


In [17]:
def merge_pair(tokens, pair, new_token):
    new_tokens = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return new_tokens


def bpe_encode(text):
    tokens = list(text)

    for pair, new_token in merges.items():
        tokens = merge_pair(tokens, pair, new_token)

    return [bpe_stoi[token] for token in tokens]


def bpe_decode(token_ids):
    return "".join(bpe_itos[token_id] for token_id in token_ids)

In [18]:
sample_text = "The quarrel between Agamemnon and Achilles."

encoded = bpe_encode(sample_text)
decoded = bpe_decode(encoded)

print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)
print("Exact match:", sample_text == decoded)

Original: The quarrel between Agamemnon and Achilles.
Encoded: [789, 364, 100, 66, 855, 162, 321, 818, 922, 92, 407, 118, 7]
Decoded: The quarrel between Agamemnon and Achilles.
Exact match: True


In [20]:
print("Total characters:", len(text))
print("First 100 characters:", repr(text[:100]))
print("Last 100 characters:", repr(text[-100:]))

Total characters: 1420167
First 100 characters: 'BOOK I.\nThe quarrel between Agamemnon and Achilles—Achilles withdraws from the war, and sends his mo'
Last 100 characters: 'form and voice of Mentor, and presently made a covenant of peace between the two contending parties.'


In [21]:
bpe_encoded = bpe_encode(text)

bpe_data = torch.tensor(bpe_encoded, dtype=torch.long)

n = int(0.9 * len(bpe_data))

bpe_train_data = bpe_data[:n]
bpe_val_data = bpe_data[n:]

print("Total BPE tokens:", len(bpe_data))
print("Training tokens:", len(bpe_train_data))
print("Validation tokens:", len(bpe_val_data))
print("BPE vocabulary size:", bpe_vocab_size)

Total BPE tokens: 424914
Training tokens: 382422
Validation tokens: 42492
BPE vocabulary size: 1080
